In [ ]:
""""
explore THOR tracea
"""

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import scipy
import pywt

In [ ]:
data_root = 'data'
plots_root = 'traces'

mV_per_ADC = 1000. / 4096.
bits = 8
tstep = 12e-9  # Sampling rate in seconds (40 MHz)


In [ ]:
all_files = os.listdir(data_root)
all_traces = []

for file in all_files:
    read_path = os.path.join(data_root, file)
    print(read_path)
    # data = np.loadtxt(read_path, skiprows=1)
    adc = np.loadtxt(read_path, dtype=np.int32, delimiter=',')
    time = tstep * np.arange(adc.size)
    
    all_traces.append(adc)

    plt.figure()
    plt.plot(time, adc)

    write_file = file.replace('.xtr', '.png')
    write_path = os.path.join(plots_root, write_file)
    plt.savefig(write_path)
    plt.close()

In [ ]:
plt.figure(figsize=(10, 3), dpi=200)
for trace, file in zip(all_traces, all_files):
    plt.plot(trace, alpha=.3, label=file)
# plt.legend()


In [ ]:
i = 0
print(len(all_traces))

In [ ]:
plt.figure(figsize=(10, 2), dpi=200)
i += 1
print(i, all_files[i])
plt.plot(all_traces[i], alpha=.3)

In [ ]:
# TODO try NNLSR on these
# Also do a peak pulse comparison on NaI_trace_filtered_220726_045157_buffer_0.txt like these files
# Reconstruct other THOR trace using sim method...

# Would be nice to make outlier filter based on FWHM or area under curve, but interference bewteen values is getting in the way...

# Idea: estimate event pulse as a collection of noisy observations, where is the noise is somehow scaled to the relative bit precision of that event
#     Should quantization be treated as uniform distribution or what?
#     pulse should also be smooth...
#     also they may still have small time shifts...

# Idea better peak finding method... : xcorr(trace, pulse[peak-5:peak+5]) or something similar

# Try weighted clustering (weighted to peaks) of normalized local extrema

# fit multivariable gaussian?

# Expectation Maximization! Init one class to approximately kernel shape. Init others to... random

# Another good idea: need to do deconvolution with some sort of saturation mask, where maked values in both weights and trace do not contribute to the deconv

# Another idea: This could be solved with NMF, similar to

In [ ]:
dist_to_peak = 13 # TODO might need to change these
kernel_len = 100
trim = True
good_traces = [18,23]
ok_traces = [0,1,2]
all_ok = [0,1,2,18,23]
all = np.arange(len(all_traces))

peaks_and_areas = []

fig1, axes1 = plt.subplots(1, 1, figsize=(14, 3), dpi=200)
fig, axes = plt.subplots(1, 2, figsize=(14, 3), dpi=200)
fig2, axes2 = plt.subplots(1, 1, figsize=(14, 4), dpi=200)

count = 0

for index in all:
    trace = all_traces[index]
    # peaks = scipy.signal.find_peaks(trace, prominence=10)[0]
    sharpened = np.convolve(trace, np.array([-1,100,-1]), mode='same')
    peaks = scipy.signal.find_peaks(sharpened,
                                    distance=3,
                                    # threshold=2 #useless because must be greater than 1, but 2 only gives 2 peaks in entire set
                                    prominence=10,
                                    # plateau_size=[0,2]
                                    )[0]
    
    trace_mode = scipy.stats.mode(trace, keepdims=False)[0]
    zero_baseline = trace - trace_mode
    
    axes1.plot(zero_baseline)
    
    for peak in peaks:
        event = zero_baseline[peak-dist_to_peak: peak-dist_to_peak+kernel_len]
        event_index = np.arange(event.size)
        
        if event.size < 1 or zero_baseline[peak] < 1:
            continue
            
        mx = np.max(event)
        if (
            np.sum(event == mx) > 1
            or np.sum(event < -2) > 3
            # or peak < 10
            # or np.where(event/np.max(event) == 1)[0] < dist_to_peak
            # or not np.all(np.diff(event) > 0) # looking for bad cuts that
        ):
            continue  
            
        count += 1
        
        # trim event if values start rising again...
        # if issues, do the above  type filters again to narrow down cause
        if trim:
            dtrace_dt_gt0 = np.where(np.diff(event[dist_to_peak+1:]) > 0)[0] + dist_to_peak+1
            if dtrace_dt_gt0.size > 0:
                cut = dtrace_dt_gt0[0]
                event = event[:cut]
                event_index = event_index[:cut]

            dtrace_dt_lt0 = np.where(np.diff(event[:dist_to_peak-1]) < 0)[0]
            if dtrace_dt_lt0.size > 0:
                cut = dtrace_dt_lt0[-1] + 1 # shift 1 for diff, exclusive upper bound
                event = event[cut:]
                event_index = event_index[cut:]
                
        if event.size < 50:
            continue
        
        axes1.plot(peak, zero_baseline[peak], marker='*', linestyle='')
        
        color = matplotlib.colors.to_hex([zero_baseline[peak]/255,0,(255-zero_baseline[peak])/255])
        axes[0].plot(event_index, event, color=color, alpha=.2)
        axes[0].plot(dist_to_peak, zero_baseline[peak], '*', color='green', alpha=.2)
        axes[1].plot(event_index, event/np.max(event),  color=color, alpha=.1)
        
        padded = np.zeros(kernel_len * 2)
        padded[:event.size] = event/np.max(event)
        fft = scipy.fft.fft(padded)
        fft = np.abs(fft)
        axes2.plot(fft[:fft.size//2], color=color, alpha=.3)

axes[0].set_title('{} Isolated Peaks'.format(count))
axes[1].set_title('{} Normed Isolated Peaks'.format(count))
axes[1].set_ylim(0, 1.1)

# axes2.set_yscale('log')
axes2.set_xlim(0, 100)

# axes1.set_xlim(13000, 15500)

# TODO try enforcing wide "prominnence between minima", with low prominence...
        

In [ ]:
trace = all_traces[18]

# lower = 0
# upper = trace.size
lower = 13000
upper = 15500

section = trace[lower:upper]

fig, axes = plt.subplots(1, 1, figsize=(14, 3), dpi=200)
axes.plot(section, label='Trace')

peaks = scipy.signal.find_peaks(section,
                                # distance=3,
                                # threshold=2 #useless because must be greater than 1, but 2 only gives 2 peaks in entire set
                                prominence=2,
                                # plateau_size=[0,2],
                                )[0]
plt.plot(peaks, section[peaks], linestyle='', marker='*', label='Peaks')
    

from util.Processing import td_nnlsr_deconvolve
from util.DataGen import nai_pulse
time, kernel = nai_pulse(1)
deconv = td_nnlsr_deconvolve(section, kernel=kernel)
axes.plot(deconv, label='Deconv')

corr = np.correlate(section, kernel, mode='same')
axes.plot(corr, label='Kernel XCorr')

kernel_peak = np.argmax(kernel)
peak_width = 3 # < 7
conv_kernel = kernel[kernel_peak-peak_width : kernel_peak + peak_width]
# corr = np.correlate(section, conv_kernel, mode='same')
corr = scipy.signal.correlate(section, conv_kernel, mode='same')
axes.plot(corr, label='Kernel Peak XCorr')

axes.legend()